In [ ]:
import math

def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

def get_luminance(rgb):
    r, g, b = [x / 255.0 for x in rgb]
    r = r / 12.92 if r <= 0.03928 else ((r + 0.055) / 1.055) ** 2.4
    g = g / 12.92 if g <= 0.03928 else ((g + 0.055) / 1.055) ** 2.4
    b = b / 12.92 if b <= 0.03928 else ((b + 0.055) / 1.055) ** 2.4
    return 0.2126 * r + 0.7152 * g + 0.0722 * b

def get_contrast_ratio(hex1, hex2):
    lum1 = get_luminance(hex_to_rgb(hex1))
    lum2 = get_luminance(hex_to_rgb(hex2))
    lighter = max(lum1, lum2)
    darker = min(lum1, lum2)
    return (lighter + 0.05) / (darker + 0.05)

def find_accessible_color(fg_hex, bg_hex, target_ratio=4.5, adjust_fg=True):
    # Simplistic iterative approach
    ratio = get_contrast_ratio(fg_hex, bg_hex)
    if ratio >= target_ratio:
        return fg_hex, bg_hex, ratio
    
    fg_rgb = list(hex_to_rgb(fg_hex))
    bg_rgb = list(hex_to_rgb(bg_hex))
    
    # Try adjusting significantly to find a safe value
    for i in range(100):
        if adjust_fg:
            # Lighten FG
             fg_rgb = [min(255, c + 2) for c in fg_rgb]
        else:
            # Darken BG
            bg_rgb = [max(0, c - 2) for c in bg_rgb]
            
        new_fg = '#{:02x}{:02x}{:02x}'.format(*fg_rgb)
        new_bg = '#{:02x}{:02x}{:02x}'.format(*bg_rgb)
        new_ratio = get_contrast_ratio(new_fg, new_bg)
        
        if new_ratio >= target_ratio:
            return new_fg, new_bg, new_ratio
            
    return fg_hex, bg_hex, ratio

# Colors from styles.css
bg_dark = "#1a1a1a"
bg_gradient_end = "#2a2420"
text_meta = "#c8c3be"
text_body = "#dcd7d2"
text_h1 = "#f2f0ed"
text_price = "#b8d4ff"

# Buttons
btn_paypal_bg = "#003087"
btn_revolut_bg = "#335DFF"
btn_offline_bg = "#444444" # Expanded from #444
btn_text = "#ffffff"

print("--- Contrast Check ---")

# 1. Check text color on details/p
print(f"Meta Text ({text_meta}) on BG Dark ({bg_dark}): {get_contrast_ratio(text_meta, bg_dark):.2f}")
print(f"Meta Text ({text_meta}) on BG Gradient End ({bg_gradient_end}): {get_contrast_ratio(text_meta, bg_gradient_end):.2f}")

print(f"Body Text ({text_body}) on BG Dark ({bg_dark}): {get_contrast_ratio(text_body, bg_dark):.2f}")

# 2. Check Link Color (none explicit found in CSS, checking likely defaults or Price)
print(f"Price Text ({text_price}) on BG Dark ({bg_dark}): {get_contrast_ratio(text_price, bg_dark):.2f}")

# 3. Check Button Color
print(f"PayPal Button Text ({btn_text}) on BG ({btn_paypal_bg}): {get_contrast_ratio(btn_text, btn_paypal_bg):.2f}")
print(f"Revolut Button Text ({btn_text}) on BG ({btn_revolut_bg}): {get_contrast_ratio(btn_text, btn_revolut_bg):.2f}")
print(f"Offline Button Text ({btn_text}) on BG ({btn_offline_bg}): {get_contrast_ratio(btn_text, btn_offline_bg):.2f}")

print("\n--- Suggestions ---")
# Check and suggest
pairs = [
    ("Meta Text", text_meta, bg_dark, True),
    ("Meta Text (Grad)", text_meta, bg_gradient_end, True), 
    ("Revolut Button", btn_text, btn_revolut_bg, False) # Usually adjust BG for buttons
]

for name, fg, bg, adjust_fg in pairs:
    ratio = get_contrast_ratio(fg, bg)
    if ratio < 4.5:
        new_fg, new_bg, new_ratio = find_accessible_color(fg, bg, 4.5, adjust_fg)
        if adjust_fg:
            print(f"Fix {name}: Change color to {new_fg} (ratio {new_ratio:.2f})")
        else:
            print(f"Fix {name}: Change background to {new_bg} (ratio {new_ratio:.2f})")
